# CausalDiscovery：Mistral-7B 因果检测与抽取基线

与 KAPipe notebook 一样，读取本项目清洗后的数据，生成逐样本预测，并进入同一个 `src.evaluator.Evaluator`。

**当前默认配置：Mistral-7B-Instruct-v0.3；Detection = FICL；Extraction = CoT；BF16；adapted/stabilized v2。**

作者 prompt 和流程仍是方法主体。本适配版根据第一次 smoke 的失败模式增加了最小输出约束：Detection 只返回一个 JSON；Extraction 必须复制原文连续 span、把全部关系放入一个平铺 JSON。两阶段均显式使用 greedy decoding，解析器只读取首个完整对象及紧邻的逗号分隔对象，避免吸收模型续写的伪造 `Input/Text` 示例。它不是作者原配置的一字不改复现，论文中应标为 **Anuyah et al. (2025) — Mistral-7B (adapted, BF16)**。

CoT 只是 `chain_of_thought` prompt 中要求模型静默分析的文字，不是模型级 thinking mode。本 notebook 不启用独立 reasoning/thinking 开关。

数据流为：原始 text → 作者 prompt + 最小适配规则 → Detection → 预测正例 Extraction → 稳健 JSON 转换 → 全样本 evaluator。默认每个数据集只跑前 10 条 smoke；全量关闭。旧的 `mistral7b_ficl_cot_v1` 4-bit 结果保留，新配置使用独立运行 ID，不会覆盖旧文件。


In [ ]:
from pathlib import Path
import logging
import sys

# 选择独立的 Python (CausalDiscovery) kernel；可从项目根目录或 notebooks 目录启动。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "evaluator.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请在 Master thesis 项目内启动 notebook。")
if Path(sys.prefix).name.lower() != "causaldiscovery":
    raise RuntimeError(f"当前解释器为 {sys.executable}，请切换到 Python (CausalDiscovery) kernel。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("causal_discovery_notebook")

SOURCE_DIR = ROOT / "reference code from related work" / "CausalDiscovery-main" / "CausalDiscovery-main"
MODEL_DIR = (ROOT / "reference code from related work" / "CausalDiscovery-model"
             / "Mistral-7B-Instruct-v0.3")
MODEL_NAME_OR_PATH = str(MODEL_DIR)

DATASET_NAMES = ["cnc_sft_test", "li", "ade", "politicause"]
BATCH_SIZE = 4
PRIMARY_METRIC = None  # 使用 evaluator 的数据集默认值，同时报告 Strict 和 Anchor。
RUN_ID = "mistral7b_bf16_ficl_cot_adapted_v2"
OUTPUT_DIR = ROOT / "results" / "eval_report" / "causal_discovery_baseline"
REUSE_COMPLETED_STAGES = True

RUN_SMOKE = True
EVAL_SAMPLE_N = 10
RUN_FULL_EVAL = False
FULL_EVAL_SAMPLE_N = None

logger.info("项目：%s；解释器：%s", ROOT, sys.executable)

## 1. 运行条件

本机使用独立 conda 环境 `CausalDiscovery` 和 Jupyter kernel `Python (CausalDiscovery)`。实际 Python、PyTorch、Transformers、Accelerate 等版本会写入每次运行的 manifest，不会影响项目其他 notebook 的环境。

模型文件位于 `D:/Master thesis/reference code from related work/CausalDiscovery-model/Mistral-7B-Instruct-v0.3`。本 notebook 使用三个 `model-*.safetensors` 分片和 index；不读取重复的 `consolidated.safetensors`，也不使用 GGUF。

BF16 在模型加载时直接使用官方权重，不经过 bitsandbytes 量化。当前 RTX 4090 支持 BF16；preflight 会同时检查 CUDA、BF16、Accelerate 和必需模型文件，不会安装依赖或下载模型。


In [ ]:
import json
import pandas as pd
import torch
from IPython.display import Markdown, display
from src.data_io import DATASET_FILES, load_dataset
from src.evaluator import primary_metric_for_dataset
from src.causal_discovery_baseline import (
    ADAPTED_PROFILE, BF16, AuthorRunner, RunConfig, build_prompts,
    load_author_templates, run_causal_discovery_baseline, software_versions,
)

CONFIG = RunConfig(
    model_name_or_path=MODEL_NAME_OR_PATH,
    batch_size=BATCH_SIZE,
    profile=ADAPTED_PROFILE,
    precision=BF16,
    detection_max_new_tokens=32,
    extraction_max_new_tokens=1024,
)
TEMPLATES = load_author_templates(SOURCE_DIR)
versions = software_versions()
display(pd.DataFrame([{"Package": name, "Version": version or "NOT INSTALLED"}
                      for name, version in versions.items()]))
CUDA_AVAILABLE = torch.cuda.is_available()
BF16_SUPPORTED = torch.cuda.is_bf16_supported() if CUDA_AVAILABLE else False
logger.info("CUDA 可用：%s；BF16 支持：%s；torch CUDA：%s",
            CUDA_AVAILABLE, BF16_SUPPORTED, torch.version.cuda)
if CUDA_AVAILABLE:
    logger.info("GPU：%s；显存 %.1f GiB", torch.cuda.get_device_name(0),
                torch.cuda.get_device_properties(0).total_memory / 1024**3)
else:
    logger.warning("当前 kernel 无法使用 CUDA；请切换到 Python (CausalDiscovery)。")
display(pd.DataFrame([
    {"Stage": "Detection", "Prompt": CONFIG.detection_prompt,
     "Model thinking": "Disabled", "Precision": "BF16 (unquantized)",
     "Decoding": "greedy; do_sample=False", "Max new tokens": CONFIG.detection_max_new_tokens},
    {"Stage": "Extraction", "Prompt": CONFIG.extraction_prompt,
     "Model thinking": "Disabled; CoT instruction is inside the prompt",
     "Precision": "BF16 (unquantized)", "Decoding": "greedy; do_sample=False",
     "Max new tokens": CONFIG.extraction_max_new_tokens},
]))


def require_inference_environment() -> None:
    '''正式推理前核对 CUDA、BF16、依赖和本地模型文件；不会执行安装或下载。'''
    missing = [name for name in ("accelerate",) if not versions[name]]
    required_model_files = [
        "config.json", "model.safetensors.index.json", "tokenizer.json", "tokenizer_config.json",
        "model-00001-of-00003.safetensors", "model-00002-of-00003.safetensors",
        "model-00003-of-00003.safetensors",
    ]
    missing_model_files = [name for name in required_model_files if not (MODEL_DIR / name).is_file()]
    if not CUDA_AVAILABLE or not BF16_SUPPORTED or missing or missing_model_files:
        raise RuntimeError(f"推理环境未就绪：CUDA={CUDA_AVAILABLE}；BF16={BF16_SUPPORTED}；"
                           f"缺少依赖={missing}；缺少模型文件={missing_model_files}。"
                           "只检查 notebook 时可关闭 RUN_SMOKE。")
    import accelerate
    logger.info("推理依赖已导入：accelerate=%s", accelerate.__version__)


## 2. 数据与 prompt 核对

使用项目正式数据入口：CNC 固定 test (`cnc_sft_test`)、Li 全数据集、ADE 当前 test、PolitiCAUSE 当前 test。下表统计只用于核对；gold 标签和关系不会进入模型 prompt。

输入只把 `text` 映射为作者 CSV 的 `sentence`。适配规则不包含数据集标签或示例答案，也不读取 gold。下面同时展示最终 Detection/Extraction prompt，便于确认新增的 JSON 停止要求和原文 span 要求。


In [ ]:
if EVAL_SAMPLE_N is None or EVAL_SAMPLE_N < 1:
    raise ValueError("Smoke 的 EVAL_SAMPLE_N 必须是正整数。")
if FULL_EVAL_SAMPLE_N is not None and FULL_EVAL_SAMPLE_N < 1:
    raise ValueError("FULL_EVAL_SAMPLE_N 必须为 None 或正整数。")

DATASETS = {name: load_dataset(name) for name in DATASET_NAMES}
dataset_rows = []
for name, samples in DATASETS.items():
    dataset_rows.append({
        "Dataset": name, "Input": DATASET_FILES[name], "N": len(samples),
        "Positive": sum(bool(s["has_causal"]) for s in samples),
        "Gold pairs": sum(len(s.get("relations", [])) for s in samples),
        "Primary extraction metric": PRIMARY_METRIC or primary_metric_for_dataset(name),
    })
display(pd.DataFrame(dataset_rows))

preview_sample = DATASETS[DATASET_NAMES[0]][0]
for stage in ("detection", "extraction"):
    prompt = build_prompts([preview_sample], stage, TEMPLATES, CONFIG)[0]
    display(Markdown(f"### {stage.title()} — 原始模板预览"))
    display(Markdown("```text\n" + prompt + "\n```"))
logger.info("4 份作者源码的 SHA-256 已读取；实际运行会将它们写入 manifest。")

## 3. 小样本运行

每个数据集读取前 `EVAL_SAMPLE_N` 条。先对全部样本检测，只对预测正例抽取，再交给同一个 evaluator。漏检、误报、格式错误和非原文 span 均保留并统计。

两阶段使用本地官方 Transformers 权重的 BF16，不进行 4-bit 量化。Extraction 上限为 1024 tokens：真实数据中 CNC 最多 5 对关系、ADE 最多 10 对、Li 最多 12 对；按作者平铺 schema 和本地 tokenizer 估算的最长 gold 输出约 818 tokens，因此保留到 1024。RTX 4090 的 24 GiB 显存足以运行当前 batch 4；若出现显存不足，只减小 `BATCH_SIZE`，不要改变 prompt、解码或精度。两阶段显式使用 `do_sample=False`，因此不再依赖 Transformers pipeline 的采样默认值。

适配解析器会恢复首个完整 JSON，也允许模型把多对关系误写为紧邻的逗号分隔 JSON；一旦出现新的 `Input:` 或 `Text:`，后续内容不会作为当前样本预测。manifest 会记录 BF16、prompt 已修改、软件版本和实际生成配置。


In [ ]:
def result_row(result: dict) -> dict:
    '''汇总统一 evaluator 的两种抽取口径，避免只展示 detected-only。'''
    report = result["report"]
    metric = report["extraction"]["primary_metric"]
    extraction = report["extraction"][metric]
    diagnostics = report["baseline"]["diagnostics"]
    return {
        "Dataset": result["dataset"], "N": result["n_samples"],
        "Detection F1": report["detection"]["f1"], "Extraction metric": metric,
        "Extraction F1 (all)": extraction["all_samples"]["f1"],
        "Extraction F1 (detected-only)": extraction["detected_only"]["f1"],
        "Detection parse errors": diagnostics["invalid_detection_outputs"],
        "Extraction parse errors": diagnostics["invalid_extraction_outputs"],
        "Normalized detection outputs": diagnostics["normalized_detection_outputs"],
        "Normalized extraction outputs": diagnostics["normalized_extraction_outputs"],
        "Report": str(result["paths"]["report_md"]),
    }


def run_selected_datasets(phase: str, sample_n: int | None) -> list[dict]:
    '''复用作者 runner，并将每个数据集的完整结果保存到独立目录。'''
    require_inference_environment()
    results = []
    for dataset in DATASET_NAMES:
        samples = DATASETS[dataset] if sample_n is None else DATASETS[dataset][:sample_n]
        logger.info("开始 %s / %s：%s 条", phase, dataset, len(samples))
        runner = AuthorRunner(SOURCE_DIR, CONFIG)
        result = run_causal_discovery_baseline(
            samples=samples, runner=runner, dataset=dataset,
            output_dir=OUTPUT_DIR / phase / dataset,
            run_name=f"{RUN_ID}_{dataset}_n{len(samples)}",
            primary_metric=PRIMARY_METRIC,
            reuse_completed_stages=REUSE_COMPLETED_STAGES,
        )
        results.append(result)
        display(Markdown("```text\n" + result["formatted_report"] + "\n```"))
    return results


smoke_results = []
if RUN_SMOKE:
    smoke_results = run_selected_datasets("smoke", EVAL_SAMPLE_N)
    display(pd.DataFrame([result_row(result) for result in smoke_results]))
else:
    logger.info("Smoke 已关闭；未加载模型。")


## 4. 全量运行

先确认新 BF16 adapted smoke 的原始输出、格式失败率和 span 对应明显改善，再把顶部 `RUN_FULL_EVAL=True`、`FULL_EVAL_SAMPLE_N=None`，重新运行配置单元与本单元。正式结果与 smoke 保存在不同目录，但使用同一 `RUN_ID` 表明配置一致。

BF16 需要约 14–15 GiB 权重显存。若显存不足，可减小 `BATCH_SIZE` 后改用新的 `RUN_ID`；不要在同一运行 ID 下混用 batch 或精度。当前不建议回退 4-bit，因为本轮目标之一就是分离量化误差。


In [ ]:
full_results = []
if RUN_FULL_EVAL:
    full_results = run_selected_datasets("full", FULL_EVAL_SAMPLE_N)
    display(pd.DataFrame([result_row(result) for result in full_results]))
else:
    logger.info("全量评估已关闭。开启方式：RUN_FULL_EVAL=True，FULL_EVAL_SAMPLE_N=None。")

## 5. 输出与复核

每个数据集目录保存：

| 文件后缀 | 内容 |
|---|---|
| `.input.csv` | 全部输入的 id 与 sentence，不含 gold |
| `.extraction_input.csv` | 仅检测预测正例的 id 与 sentence |
| `.detection.raw.jsonl` / `.extraction.raw.jsonl` | 逐样本模型原始续写与耗时 |
| `.predictions.jsonl` | evaluator 格式的 has_causal、triples 和解析错误 |
| `.report.json` / `.report.md` | Detection、Strict/Anchor 的 all-samples 与 detected-only 结果 |
| `.manifest.json` | 输入与作者源码哈希、适配 profile、BF16、prompt 修改状态、软件和生成配置 |

`REUSE_COMPLETED_STAGES=True` 只复用输入、作者源码、适配器源码、配置和依赖指纹均一致的完整阶段。新配置使用 `mistral7b_bf16_ficl_cot_adapted_v2`，不会读取或覆盖旧的 `mistral7b_ficl_cot_v1` 4-bit 输出。

解析器保留 cause/effect 方向和所有编号关系。适配版可以忽略首个完整 JSON 后的模型续写，并恢复紧邻的逗号分隔对象；被恢复的数量写入 `normalized_detection_outputs` 和 `normalized_extraction_outputs`。没有完整 JSON 的输出仍保留为解析错误，不会由代码猜测关系。

**论文口径：**这是作者 FICL detection + CoT extraction 流程在本项目上的 BF16 稳定化适配。必须同时报告 detection、all-samples extraction、格式失败率和具体适配内容，不能把结果写成作者原始 4-bit 配置的一字不改复现。detected-only 仅作为 span 诊断。


In [ ]:
# 查看最后一次结果的路径和前几条原始输出；不会重新加载模型。
recent_results = full_results or smoke_results
if recent_results:
    first = recent_results[0]
    display(pd.DataFrame([{"File": name, "Path": str(path)} for name, path in first["paths"].items()]))
    for stage in ("detection", "extraction"):
        raw_path = first["paths"][f"{stage}_raw"]
        with raw_path.open(encoding="utf-8") as file:
            rows = [json.loads(line) for _, line in zip(range(3), file)]
        display(Markdown(f"### {stage.title()}：原始输出前 3 条"))
        display(pd.DataFrame(rows))
else:
    logger.info("尚未运行模型预测。预期输出目录：%s", OUTPUT_DIR)
